# Transformers Hands-On Lab

This notebook demonstrates the three Transformer families and compares a local LLM (Ollama) with hosted assistants.

| Section | Architecture | Task |
|---|---|---|
| 1 | Encoder-decoder | Machine translation |
| 2 | Decoder-only | Text generation |
| 3 | Encoder-only | Sentiment analysis |
| 4 | Decoder-only (LLM via Ollama) | Prompting and comparison with ChatGPT / Claude |

In [23]:
!pip install -q "transformers[torch]" sentencepiece accelerate langchain-ollama

## 1. Encoder-decoder (machine translation)

The original Transformer design: the **encoder** reads the input sentence, and the **decoder** generates the output while using **cross-attention** to look at the encoder's output.

### 1a. BERT2BERT (English → German)

In [24]:
from transformers import AutoTokenizer, EncoderDecoderModel, AutoModelForSeq2SeqLM

model_name = "google/bert2bert_L-24_wmt_en_de"
tokenizer = AutoTokenizer.from_pretrained(model_name, pad_token="<pad>", eos_token="</s>", bos_token="<s>")
model = EncoderDecoderModel.from_pretrained(model_name, device_map="auto")

input_text = "Plants create energy through a process known as photosynthesis."
inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).to(model.device).input_ids
outputs = model.generate(inputs)[0]
print("Translated text:", tokenizer.decode(outputs, skip_special_tokens=True))

Loading weights:   0%|          | 0/1019 [00:00<?, ?it/s]

Translated text: Pflanzen erzeugen Energie durch einen Prozess, der Photosynthese genannt wird.


### 1b. T5 (English → French)

In [25]:
tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-base", device_map="auto")

batch = tokenizer("translate English to French: I love machine learning", return_tensors="pt").to(model.device)
output = model.generate(**batch, max_length=30)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

J'aime l'apprentissage par machine


### 1c. Flan-T5 fine-tuned for English → Hinglish

In [26]:
model_name = "rvv-karma/English2Hinglish-Flan-T5-Base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

batch = tokenizer("life is crazy. Not me", return_tensors="pt")
output_ids = model.generate(**batch)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Life crazy haiNot me


## Encoder-decoder observations
Translations: BERT2BERT (English→German) gave a fluent, accurate sentence: "Pflanzen erzeugen Energie durch einen Prozess, der Photosynthese genannt wird." T5 (English→French) gave "J'aime l'apprentissage par machine.", which is understandable but literal; the standard term is "apprentissage automatique". BERT2BERT was the best of the three, with T5 close behind.

Hinglish: This was the weakest. The output "Life crazy haiNot me" is garbled: words are glued together, "Not me" is left in English, and only "hai" was converted. The first two models were trained on large translation datasets, while the Hinglish model is a small fine-tune, so translation quality depends heavily on training data and model size.

## Decoder-only observations
GPT-2 continued "The future of AI is" with grammatical, fluent English: "...already in question. As AI researchers come to realize how much work AI can do in the next 20 years...". The content is vague and says little, which fits a small model that only predicts likely next tokens without understanding or fact-checking. The output stops mid-sentence because it reached the 30-token limit (max_new_tokens). This matches masked self-attention: each token is generated using only the tokens before it, one at a time.

## Encoder-only observations
The two clear sentences were classified correctly with very high confidence (POSITIVE 0.9999, NEGATIVE 0.9998). The neutral sentence "The product arrived on Tuesday." was labelled POSITIVE with 0.94 confidence, which is not sensible. The model was fine-tuned on SST-2, a binary dataset of movie reviews with only POSITIVE and NEGATIVE labels, so it has no neutral option and is forced to pick one. It is also out of domain (delivery text, not movie reviews). This shows that a high confidence score does not guarantee a correct prediction.

##

## 2. Decoder-only (text generation)

GPT-style models only have the decoder stack with **masked self-attention**: each token can only attend to earlier tokens, so the model simply predicts the next token, over and over.

In [27]:
from transformers import pipeline

generator = pipeline(task="text-generation", model="gpt2")
output = generator("The future of AI is", max_new_tokens=30, num_return_sequences=1)
print(output[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The future of AI is a lot more interesting than the future of the internet. It's not just about computers, it's about the ability to create, modify, transform,


Try changing the prompt, `max_new_tokens`, or adding `do_sample=True, temperature=0.8` and compare.

**Observations (fill in):**

> _your notes here_

## 3. Encoder-only (sentiment analysis)

Encoder-only models (BERT family) read the whole sentence at once with bidirectional self-attention, and produce representations that are used for understanding tasks such as classification, search and clustering.

In [28]:
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

sentences = [
    "I loved this movie, it was fantastic!",
    "This was the worst service I have ever had.",
    "The product arrived on Tuesday.",
]
for s in sentences:
    print(s, "->", classifier(s))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

I loved this movie, it was fantastic! -> [{'label': 'POSITIVE', 'score': 0.9998804330825806}]
This was the worst service I have ever had. -> [{'label': 'NEGATIVE', 'score': 0.9997978806495667}]
The product arrived on Tuesday. -> [{'label': 'POSITIVE', 'score': 0.9428859949111938}]


**Observations (fill in):** Was the neutral sentence classified sensibly? Why might a two-class model struggle with it?

> _your notes here_

## 4. Running LLMs locally with Ollama

**Before running this section**, Ollama must be installed and running with a model pulled:

```bash
ollama serve          # if not already running
ollama pull mistral
ollama list
```

In [29]:
!apt-get install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [34]:
!nvidia-smi

Thu Oct  1 18:47:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             31W /   70W |    9611MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [30]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull mistral

In [31]:
!ollama list

NAME              ID              SIZE      MODIFIED               
mistral:latest    6577803aa9a0    4.4 GB    Less than a second ago    


In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)
model = OllamaLLM(model="mistral")
chain = prompt | model

print(chain.invoke({"question": "Who is Sir Isaac Newton?"}))

 Sir Isaac Newton was an English mathematician, physicist, and scientist who is widely recognized as one of the most influential figures in the scientific revolution of the 17th century. He made significant contributions to various scientific fields, including mathematics, physics, and astronomy. Some of his most notable work includes the development of calculus, the laws of motion, and the universal law of gravitation. He is best known for his work "Philosophiæ Naturalis Principia," which contains his three laws of motion and the law of universal gravitation.


In [35]:
!ollama ps

NAME              ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
mistral:latest    6577803aa9a0    5.0 GB    100% GPU     4096       2 minutes from now    


### Comparison with ChatGPT / Claude

Run the same prompts on the local model and on a hosted assistant, then fill in the table in the next cell.

In [33]:
test_prompts = [
    "Explain self-attention in two sentences.",
    "Write a Python function that checks whether a string is a palindrome.",
    "A train travels 120 km in 1.5 hours. What is its average speed? Show your reasoning.",
    "Summarize the plot of Romeo and Juliet in three sentences.",
]

local_answers = {}
for q in test_prompts:
    local_answers[q] = chain.invoke({"question": q})
    print("Q:", q)
    print("A:", local_answers[q])
    print("-" * 60)

Q: Explain self-attention in two sentences.
A:  Self-attention, in the context of deep learning, is a mechanism that allows a model to weigh the importance of different words or elements within a sequence when computing the representation of each element. Essentially, it enables the model to focus on relevant information while ignoring irrelevant details, which improves the model's understanding and analysis of the data.
------------------------------------------------------------
Q: Write a Python function that checks whether a string is a palindrome.
A:  A palindrome is a word, phrase, number, or other sequence of characters that reads the same forward and backward. Here's a simple Python function to check if a given string is a palindrome. This function assumes the input string only contains alphanumeric characters and is case-insensitive.

```python
def is_palindrome(s):
    s = s.lower()
    reversed_s = s[::-1]
    return s == reversed_s
```

You can use this function like this:


### Comparison table (fill in after running)

Paste the hosted model's answer for each prompt, then rate both (1-5) on correctness, clarity and speed.

| Prompt | Mistral (Ollama) | ChatGPT / Claude | Notes |
|---|---|---|---|
| Self-attention explanation |Correct but vague; gave 3 sentences instead of 2 and didn't mention queries/keys/values. |"Self-attention lets each token compute a weighted mix of every token in the sequence, with weights from query-key similarity. This lets the model capture relationships between words regardless of distance, all in parallel." |Claude followed the 2-sentence limit and was more specific. |
| Palindrome function |Buggy. It compares s[left] with s[right] where right starts at the middle, not the end. For "A man, a plan, a canal: Panama" it returns False (its own comment says True), and "abba" also returns False. |cleaned = "".join(c.lower() for c in s if c.isalnum()) then return cleaned == cleaned[::-1] |Claude's version is correct and simpler. Mistral sounded confident while being wrong. |
| Speed word problem |Correct: 120 / 1.5 = 80 km/h, with clear steps. |120 km ÷ 1.5 h = 80 km/h. |Tie: easy question. |
| Romeo and Juliet summary |Gave 4 sentences instead of 3, added an invented detail ("centuries-old hatred"), and left out why Romeo thought Juliet was dead. |Two feuding families' children, Romeo and Juliet, fall in love and secretly marry. After a chain of mishaps Romeo believes Juliet is dead and kills himself; Juliet wakes and takes her own life. Their deaths finally end the feud between the families. |Claude kept to 3 sentences and stayed accurate. |

### Conclusion

A local model like Mistral costs nothing per query, keeps my data on my own machine, and works offline, but it needs hardware to run: on Colab it used the T4 GPU (about 4.8 GB of GPU memory for the 4.4 GB model), and it needed a large download before it could be used. It also made real mistakes: it ignored length instructions, produced a palindrome function that returns the wrong answer while claiming it works, and invented a plot detail. Hosted models like Claude are much larger and were more accurate and better at following instructions, but they require internet access, may have usage limits, and send my prompts to a third party. On simple reasoning (the speed problem) both did equally well, so the gap shows up on tasks where small errors matter, like code. Mistral is a decoder-only Transformer: it uses masked self-attention and generates one token at a time. Claude and ChatGPT are chat assistants of the same general kind, and ChatGPT is a decoder-only GPT model.